In [ ]:
# Basic data handling
import pandas as pd
import numpy as np

# Machine learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error

# Optional: plotting
import matplotlib.pyplot as plt


In [ ]:
# Paths (relative to 5_Model_Training/)
EMBEDDINGS_PATH = "../1_Data/hf_esm2_t30_150M_vhvl_embeddings.csv"
HIC_PATH = "../1_Data/preprocessed_data.csv"

# Load data
embeddings_df = pd.read_csv(EMBEDDINGS_PATH)
hic_df = pd.read_csv(HIC_PATH)

print("Embeddings shape:", embeddings_df.shape)
print("HIC data shape:", hic_df.shape)


In [ ]:
ID_COLUMN = "Clone name"
TARGET_COLUMN = "HIC retention time (min)"

# Merge on ID
data = embeddings_df.merge(
    hic_df[[ID_COLUMN, TARGET_COLUMN]],
    on=ID_COLUMN,
    how="inner"
)

print("Merged dataset shape:", data.shape)

In [ ]:
# Drop non-feature columns
X = data.drop(columns=[ID_COLUMN, TARGET_COLUMN])
y = data[TARGET_COLUMN]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

param_grid = {
    "ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)


In [ ]:
best_ridge = grid_search.best_estimator_
best_ridge.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = best_ridge.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test R²: {r2:.3f}")
print(f"Test RMSE: {rmse:.3f} min")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         "r--")
plt.xlabel("Measured HIC retention time (min)")
plt.ylabel("Predicted HIC retention time (min)")
plt.title("Ridge Regression: Predicted vs Actual")
plt.tight_layout()
plt.show()

Whats new? Standardize embeddings before ridge regression (very important)
Ridge regression assumes features are on comparable scales.
ESM embeddings often are not.
Why it helps

Prevents a few dimensions from dominating
Makes regularization behave properly

Expected gain

Often improves R² noticeably (sometimes +0.05–0.15)

Implementation (minimal change):